# 🏥⚖️ QLoRA Fine-tuning 14B — Médico & Abogado (Unsloth)
Fine-tuning de Qwen2.5-14B-Instruct sobre dataset médico-legal con Unsloth.
Setup: 2x T4, model parallelism, sin DDP.

In [1]:
# ─────────────────────────────────────────────
# CELDA 1 — Verificar entorno y GPUs
# ─────────────────────────────────────────────
import os
import gc
import torch

print(f"🖥️  GPUs disponibles: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    free, total = torch.cuda.mem_get_info(i)
    print(f"   GPU {i}: {props.name} — {props.total_memory / 1e9:.1f} GB total | {free/1e9:.1f} GB libres")

assert torch.cuda.device_count() >= 2, "❌ Necesitás 2 GPUs. Activá T4x2 en Settings."
print("\n✅ Entorno listo")


🖥️  GPUs disponibles: 2
   GPU 0: Tesla T4 — 15.6 GB total | 15.5 GB libres
   GPU 1: Tesla T4 — 15.6 GB total | 15.5 GB libres

✅ Entorno listo


In [1]:
# ─────────────────────────────────────────────
# CELDA 2 — Instalar dependencias (torch 2.9 compatible)
# ─────────────────────────────────────────────
import sys, os

os.system(f"{sys.executable} -m pip install -q 'unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git'")
os.system(f"{sys.executable} -m pip install -q 'transformers>=4.51.3' 'peft>=0.18.0' 'trl>=0.18.2,<=0.24.0' datasets lm-eval")

import importlib
for pkg in ["unsloth", "transformers", "trl", "peft", "accelerate", "torch"]:
    try:
        mod = importlib.import_module(pkg)
        print(f"✅ {pkg}=={mod.__version__}")
    except Exception as e:
        print(f"❌ {pkg}: {e}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.2/395.2 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.9/224.9 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 37.3 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.4/56.4 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.1/91.1 kB 8.7 MB/s eta 0:00:00
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
✅ unsloth==2026.3.4
✅ transformers==5.2.0
✅ trl==0.24.0
✅ peft==0.18.1
✅ accelerate==1.12.0
✅ torch==2.9.0+cu126


In [3]:
# ─────────────────────────────────────────────
# CELDA 4 — Cargar dataset médico + legal (multi-source)
# ─────────────────────────────────────────────
from datasets import load_dataset, concatenate_datasets

print("📦 Cargando dataset médico...")
medical = load_dataset("lavita/ChatDoctor-HealthCareMagic-100k", split="train")
medical = medical.filter(lambda x: "Chat Doctor" not in x.get("output", ""))
medical = medical.filter(lambda x: "chatdoctor" not in x.get("output", "").lower())

print("📦 Cargando datasets legales...")
ds1 = load_dataset("dzunggg/legal-qa-v1", split="train")
ds2 = load_dataset("MeeraR/legal-qa-dataset", split="train")
ds3 = load_dataset("jonathanli/law-stack-exchange", split="train")

# Normalizar columnas a question/answer
ds1 = ds1.select_columns(["question", "answer"])
ds2 = ds2.rename_column("answer", "answer_meera")
ds2 = ds2.map(lambda x: {"question": x["question"], "answer": x["answer_meera"] or x["Answer"] or ""})
ds2 = ds2.select_columns(["question", "answer"])
ds3 = ds3.map(lambda x: {"question": x["title"], "answer": x["body"] or x["text_label"] or ""})
ds3 = ds3.select_columns(["question", "answer"])

legal = concatenate_datasets([ds1, ds2, ds3])
legal = legal.filter(lambda x: len(x.get("question", "")) > 20 and len(x.get("answer", "")) > 50)

N_MEDICAL = 10_000
N_LEGAL   = 9_000  # usamos todos (~6,700, va a quedar un poco desbalanceado pero por la complejidad faltarian legales)

medical = medical.shuffle(seed=42).select(range(min(N_MEDICAL, len(medical))))
legal   = legal.shuffle(seed=42)

print(f"\n✅ Médico: {len(medical):,} ejemplos")
print(f"✅ Legal:  {len(legal):,} ejemplos")
print(f"\n🔍 Ejemplo médico:"); print(medical[0])
print(f"\n🔍 Ejemplo legal:"); print(legal[0])

📦 Cargando dataset médico...


README.md:   0%|          | 0.00/542 [00:00<?, ?B/s]

data/train-00000-of-00001-5e7cb295b9cff0(…):   0%|          | 0.00/70.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/112165 [00:00<?, ? examples/s]

Filter:   0%|          | 0/112165 [00:00<?, ? examples/s]

...

Generating test split:   0%|          | 0/1596 [00:00<?, ? examples/s]

Map:   0%|          | 0/2375 [00:00<?, ? examples/s]

Map:   0%|          | 0/638 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6755 [00:00<?, ? examples/s]


✅ Médico: 10,000 ejemplos
✅ Legal:  6,492 ejemplos

🔍 Ejemplo médico:
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'I have fatty liver disease, my liver hurting much of the day by the way, I have no credit card if this is the way it usually works-so if a credit card is needed then I must stop the contents. I have had a sharp pain in my lower right side and when I take my thumb and put it on my naval, my 2nd finger is where it hurts. the problem is that it didn t hurt al day yesterday just about 8 hours, went away then about l0', 'output': 'Hi, I cannot say in your particular case without directly examining. Appendix pain is quite variable BUT is on the right mostly. It is mostly below the naval BUT, this varies. Appendicitis IS an emergency and not at all likely to continue without causing potentially fatal rupture for over a day. Fatty liver is quite common and obviously most people with it do not have signif

In [4]:
# ─────────────────────────────────────────────
# CELDA 5 — Formatear en ChatML con system prompt
# ─────────────────────────────────────────────
from datasets import concatenate_datasets

SYSTEM_MEDICO = (
    "Sos un médico clínico experto con más de 20 años de experiencia. "
    "SIEMPRE respondés en español, nunca en inglés. "
    "Respondés preguntas médicas de forma clara, precisa y empática. "
    "Siempre recomendás consultar a un profesional de la salud para diagnósticos definitivos."
)

SYSTEM_LEGAL = (
    "Sos un abogado experto en derecho con más de 20 años de experiencia. "
    "SIEMPRE respondés en español, nunca en inglés. "
    "Respondés consultas legales de forma clara y precisa. "
    "Siempre aclarás que tu respuesta es orientativa y recomendás consultar a un abogado matriculado."
)

def format_medical(example):
    instruction = example.get("instruction", "").strip()
    inp         = example.get("input", "").strip()
    output      = example.get("output", "").strip()
    if not instruction or not output:
        return {"text": ""}
    user_msg = f"{instruction}\n{inp}".strip() if inp else instruction
    text = (
        f"<|im_start|>system\n{SYSTEM_MEDICO}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n{output}<|im_end|>"
    )
    return {"text": text}

def format_legal(example):
    question = example.get("question", "").strip()
    answer   = example.get("answer", "").strip()
    if not question or not answer:
        return {"text": ""}
    text = (
        f"<|im_start|>system\n{SYSTEM_LEGAL}<|im_end|>\n"
        f"<|im_start|>user\n{question}<|im_end|>\n"
        f"<|im_start|>assistant\n{answer}<|im_end|>"
    )
    return {"text": text}

medical_fmt = medical.map(format_medical, remove_columns=medical.column_names, num_proc=4)
legal_fmt   = legal.map(format_legal,   remove_columns=legal.column_names,   num_proc=4)

medical_fmt = medical_fmt.filter(lambda x: len(x["text"]) > 100)
legal_fmt   = legal_fmt.filter(lambda x: len(x["text"]) > 100)

dataset = concatenate_datasets([medical_fmt, legal_fmt]).shuffle(seed=42)
dataset = dataset.train_test_split(test_size=0.05, seed=42)

train_dataset = dataset["train"]
eval_dataset  = dataset["test"]

print(f"✅ Dataset combinado formateado")
print(f"📊 Train: {len(train_dataset):,} | Val: {len(eval_dataset):,}")
print(f"\n🔍 Ejemplo formateado:")
print(train_dataset[0]["text"][:600])
print("...")

Map (num_proc=4):   0%|          | 0/10000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/6492 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6492 [00:00<?, ? examples/s]

✅ Dataset combinado formateado
📊 Train: 15,667 | Val: 825

🔍 Ejemplo formateado:
<|im_start|>system
Sos un abogado experto en derecho con más de 20 años de experiencia. SIEMPRE respondés en español, nunca en inglés. Respondés consultas legales de forma clara y precisa. Siempre aclarás que tu respuesta es orientativa y recomendás consultar a un abogado matriculado.<|im_end|>
<|im_start|>user
Under what section of Canadian Human Rights Act is right to privacy addressed?<|im_end|>
<|im_start|>assistant
The punishment for murder under Constitution of India is typically community service.<|im_end|>
...


In [5]:
import torch

MODEL_ID   = "unsloth/Qwen2.5-14B-Instruct"
OUTPUT_DIR = "/kaggle/working/qwen25-14b-medlegal"

print(f"✅ Model ID: {MODEL_ID}")
print(f"✅ Output:   {OUTPUT_DIR}")


✅ Model ID: unsloth/Qwen2.5-14B-Instruct
✅ Output:   /kaggle/working/qwen25-14b-medlegal


In [6]:
# ─────────────────────────────────────────────
# CELDA 7 — Guardar dataset en disco
# ─────────────────────────────────────────────
DATA_PATH = "/kaggle/working/medlegal_formatted"
dataset.save_to_disk(DATA_PATH)
print(f"✅ Dataset guardado en {DATA_PATH}")


Saving the dataset (0/1 shards):   0%|          | 0/15667 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/825 [00:00<?, ? examples/s]

✅ Dataset guardado en /kaggle/working/medlegal_formatted


In [7]:
# ─────────────────────────────────────────────
# CELDA 8 — Pre-descargar modelo
# ─────────────────────────────────────────────
from huggingface_hub import snapshot_download
import torch

MODEL_ID = "unsloth/Qwen2.5-14B-Instruct"

print("📥 Descargando archivos del modelo a caché local...")
snapshot_download(
    repo_id=MODEL_ID,
    repo_type="model",
    ignore_patterns=["*.msgpack", "*.h5", "flax_*"],
)

print("✅ Modelo en caché")
print(f"   VRAM libre: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB / {torch.cuda.mem_get_info()[1]/1e9:.1f} GB")


📥 Descargando archivos del modelo a caché local...


Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

✅ Modelo en caché
   VRAM libre: 15.5 GB / 15.6 GB


In [8]:
train_script = '''
import os
import glob
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig
from transformers import EarlyStoppingCallback, TrainerCallback
from datasets import load_from_disk
from kaggle_secrets import UserSecretsClient
from huggingface_hub import snapshot_download, HfApi

MODEL_ID   = "unsloth/Qwen2.5-14B-Instruct"
DATA_PATH  = "/kaggle/working/medlegal_formatted"
OUTPUT_DIR = "/kaggle/working/qwen25-14b-medlegal"
HF_REPO    = "Cukinator/qwen25-14b-medlegal"

# ─────────────────────────────────────────────
CONTINUAR   = True   # ← True = retomar sesión anterior | False = entrenar desde cero
SAVE_STEPS  = 50     # ← Guardar checkpoint cada N pasos
UPLOAD_STEPS = 50    # ← Subir a HuggingFace cada N pasos (0 = desactivado)
# ─────────────────────────────────────────────

os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


# ─── Callback: sube SECUENCIALMENTE a HF cada N pasos ───────────────────────
class AutoUploadCallback(TrainerCallback):
    def __init__(self, repo_id: str, output_dir: str, token: str, every_n_steps: int = 50):
        self.repo_id      = repo_id
        self.output_dir   = output_dir
        self.token        = token
        self.every_n      = every_n_steps
        self.api          = HfApi()

    def on_save(self, args, state, control, **kwargs):
        """Se llama justo después de que el Trainer guarda un checkpoint."""
        if self.every_n <= 0:
            return
        if state.global_step % self.every_n == 0:
            print(f"📤 [paso {state.global_step}] Subiendo checkpoint SECUENCIALMENTE a HuggingFace...")
            try:
                self.api.create_repo(
                    repo_id=self.repo_id, private=True,
                    exist_ok=True, token=self.token
                )
                
                # Buscamos la carpeta del checkpoint actual
                ckpt_folder = os.path.join(self.output_dir, f"checkpoint-{state.global_step}")
                
                if os.path.exists(ckpt_folder):
                    # Recorremos los archivos uno por uno
                    for root, dirs, files in os.walk(ckpt_folder):
                        for file in files:
                            local_path = os.path.join(root, file)
                            # Calculamos la ruta relativa para mantener la estructura en el repo
                            path_in_repo = os.path.relpath(local_path, self.output_dir)
                            
                            print(f"   ➤ Subiendo {file} ...")
                            self.api.upload_file(
                                path_or_fileobj=local_path,
                                path_in_repo=path_in_repo,
                                repo_id=self.repo_id,
                                token=self.token,
                                commit_message=f"Upload {file} (step {state.global_step})"
                            )
                print(f"✅ Subida secuencial completada → https://huggingface.co/{self.repo_id}")
            except Exception as e:
                print(f"⚠️ Error al subir en paso {state.global_step}: {e}")
# ────────────────────────────────────────────────────────────────────────────


def main():
    global MODEL_ID
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")

    resume_from  = False
    load_adapter = None

    if CONTINUAR:
        print("📥 Intentando descargar checkpoint desde HuggingFace...")
        try:
            snapshot_download(repo_id=HF_REPO, local_dir=OUTPUT_DIR, token=hf_token)
            print("✅ Descarga completada")

            ckpt_dirs = sorted(
                glob.glob(os.path.join(OUTPUT_DIR, "checkpoint-*")),
                key=lambda p: int(p.split("-")[-1])
            )
            if ckpt_dirs:
                resume_from = ckpt_dirs[-1]
                print(f"♻️  Retomando desde Trainer checkpoint: {resume_from}")

            elif os.path.exists(os.path.join(OUTPUT_DIR, "trainer_state.json")):
                resume_from = OUTPUT_DIR
                print(f"♻️  Retomando desde raíz: {resume_from}")

            elif os.path.exists(OUTPUT_DIR) and any(os.scandir(OUTPUT_DIR)):
                print("⚠️  El repo HF tiene pesos LoRA pero NO un Trainer checkpoint.")
                adapter_cfg = os.path.join(OUTPUT_DIR, "adapter_config.json")
                if os.path.exists(adapter_cfg):
                    print("    Se cargará el modelo base y se restaurarán los pesos del adaptador.")
                    load_adapter = OUTPUT_DIR
                else:
                    print("    No se encontró adapter_config.json. Entrenando desde cero.")
                resume_from = False

            else:
                print("⚠️  El repo HF está vacío. Entrenando desde cero con modelo base.")
                resume_from = False

        except Exception as e:
            print(f"⚠️  No se pudo descargar desde HF ({e}).")
            print("    Entrenando desde cero con modelo base.")
            resume_from = False
    else:
        print("🆕 Entrenamiento desde cero")

    print(f"📥 Cargando modelo con Unsloth: {MODEL_ID}")
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_ID,
        max_seq_length=512,
        dtype=torch.float16,
        load_in_4bit=True,
        device_map="auto",
        max_memory={0: "12GB", 1: "12GB"},
    )

    tokenizer.pad_token = "<|endoftext|>"

    model = FastLanguageModel.get_peft_model(
        model,
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=42,
    )

    if load_adapter:
        print(f"🔁 Restaurando pesos del adaptador desde {load_adapter}...")
        bin_path = os.path.join(load_adapter, "adapter_model.bin")
        sf_path  = os.path.join(load_adapter, "adapter_model.safetensors")
        try:
            from peft import set_peft_model_state_dict
            if os.path.exists(sf_path):
                from safetensors.torch import load_file
                adapter_weights = load_file(sf_path, device="cpu")
            elif os.path.exists(bin_path):
                adapter_weights = torch.load(bin_path, map_location="cpu", weights_only=True)
            else:
                raise FileNotFoundError("No se encontró adapter_model.bin ni .safetensors")
            set_peft_model_state_dict(model, adapter_weights)
            print("✅ Pesos del adaptador restaurados correctamente")
        except Exception as e:
            print(f"⚠️  No se pudieron restaurar los pesos del adaptador ({e}). Desde cero.")

    dataset    = load_from_disk(DATA_PATH)
    train_data = dataset["train"]
    eval_data  = dataset["test"]

    print(f"📊 Train: {len(train_data):,} | Val: {len(eval_data):,}")

    training_args = SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=8,
        warmup_ratio=0.05,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        fp16=True,
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=SAVE_STEPS,          # ← evaluar cada 50 pasos
        save_strategy="steps",
        save_steps=SAVE_STEPS,          # ← guardar cada 50 pasos
        save_total_limit=4,             # ← mantener últimos 4 checkpoints
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        report_to="none",
        dataset_text_field="text",
        max_seq_length=512,
        packing=True,
        dataloader_num_workers=2,
        optim="adamw_8bit",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_data,
        eval_dataset=eval_data,
        args=training_args,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=3),
            AutoUploadCallback(           # ← subida automática a HF
                repo_id=HF_REPO,
                output_dir=OUTPUT_DIR,
                token=hf_token,
                every_n_steps=UPLOAD_STEPS,
            ),
        ],
    )

    print(f"🚀 Iniciando entrenamiento... (resume_from={resume_from})")
    trainer.train(resume_from_checkpoint=resume_from)

    print(f"💾 Guardando modelo final en {OUTPUT_DIR}...")
    model.save_pretrained(OUTPUT_DIR)
    tokenizer.save_pretrained(OUTPUT_DIR)

    print("📤 Subiendo modelo final a HuggingFace Hub...")
    api = HfApi()
    api.create_repo(repo_id=HF_REPO, private=True, exist_ok=True, token=hf_token)
    api.upload_folder(
        folder_path=OUTPUT_DIR,
        repo_id=HF_REPO,
        repo_type="model",
        token=hf_token,
    )
    print(f"✅ Modelo final subido → https://huggingface.co/{HF_REPO}")

if __name__ == "__main__":
    main()
'''

with open("/kaggle/working/train_unsloth.py", "w") as f:
    f.write(train_script)

print("✅ train_unsloth.py guardado")

✅ train_unsloth.py guardado


In [9]:
import os, shutil, gc, torch

os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
torch.cuda.empty_cache()

for i in range(torch.cuda.device_count()):
    free, total = torch.cuda.mem_get_info(i)
    print(f"   GPU {i}: {free/1e9:.1f} GB libres / {total/1e9:.1f} GB totales")

ckpt_dir = "/kaggle/working/qwen25-14b-medlegal"
if os.path.exists(ckpt_dir):
    shutil.rmtree(ckpt_dir)
    print(f"🧹 Output dir limpiado: {ckpt_dir}")

print("🚀 Lanzando entrenamiento con Unsloth...\n")
exit_code = os.system("python /kaggle/working/train_unsloth.py")

if exit_code == 0:
    print("\n✅ Entrenamiento completado exitosamente")
else:
    print(f"\n❌ Error (exit code: {exit_code}) — revisá los logs arriba")


   GPU 0: 15.5 GB libres / 15.6 GB totales
   GPU 1: 15.5 GB libres / 15.6 GB totales
🚀 Lanzando entrenamiento con Unsloth...

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
📥 Intentando descargar checkpoint desde HuggingFace...


Fetching 50 files: 100%|██████████| 50/50 [00:06<00:00,  8.21it/s]


✅ Descarga completada
♻️  Retomando desde Trainer checkpoint: /kaggle/working/qwen25-14b-medlegal/checkpoint-200
📥 Cargando modelo con Unsloth: unsloth/Qwen2.5-14B-Instruct
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights: 100%|██████████| 579/579 [00:05<00:00, 98.90it/s, Materializing param=model.norm.weight]                               
Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.3.4 patched 48 layers with 0 QKV layers, 0 O layers and 0 MLP layers.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
📊 Train: 15,667 | Val: 825


...

   ➤ Subiendo tokenizer.json ...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  ...kpoint-650/tokenizer.json: 100%|██████████| 11.4MB / 11.4MB            

Processing Files (1 / 1)      : 100%|██████████| 11.4MB / 11.4MB,   ???B/s  

  ...kpoint-650/tokenizer.json: 100%|██████████| 11.4MB / 11.4MB            

Processing Files (1 / 1)      : 100%|██████████| 11.4MB / 11.4MB,  0.00B/s  
New Data Upload               : |          |  0.00B /  0.00B,  0.00B/s  
  ...kpoint-650/tokenizer.json: 100%|██████████| 11.4MB / 11.4MB            
100%|██████████| 699/699 [9:59:15<00:00, 51.44s/it]   


   ➤ Subiendo tokenizer_config.json ...
✅ Subida secuencial completada → https://huggingface.co/Cukinator/qwen25-14b-medlegal
{'loss': '1.559', 'grad_norm': '0.1939', 'learning_rate': '6.987e-07', 'epoch': '0.9669'}
{'train_runtime': '3.596e+04', 'train_samples_per_second': '0.311', 'train_steps_per_second': '0.019', 'train_loss': '1.129', 'epoch': '1'}
💾 Guardando modelo final en /kaggle/working/qwen25-14b-medlegal...
📤 Subiendo modelo final a HuggingFace Hub...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

  .../checkpoint-550/scaler.pt: 100%|██████████| 1.38kB / 1.38kB            


  ...int-550/training_args.bin: 100%|██████████| 5.71kB / 5.71kB            



  ...eckpoint-550/scheduler.pt: 100%|██████████| 1.47kB / 1.47kB            




  ...ckpoint-550/rng_state.pth: 100%|██████████| 14.6kB / 14.6kB            





  .../checkpoint-650/scaler.pt: 100%|██████████| 1.38kB / 1.38kB            






  ...int-600/training_args.bin: 100%|██████████| 5.71kB / 5.71kB            







  ...eckpoint-650/scheduler.pt: 100%|██████████| 1.47kB / 1.47kB            








  ...int-699/training_args.bin: 100%|██████████| 5.71kB / 5.71kB            









  ...int-650/training_args.bin: 100%|██████████| 5.71kB / 5.71kB            










  .../checkpoint-600/scaler.pt: 100%|██████████| 1.38kB / 1.38kB            

  .../checkpoint-550/scaler.

✅ Modelo final subido → https://huggingface.co/Cukinator/qwen25-14b-medlegal

✅ Entrenamiento completado exitosamente


In [13]:
import torch
import gc
from unsloth import FastLanguageModel
from kaggle_secrets import UserSecretsClient

ADAPTER_DIR = "/kaggle/working/qwen25-14b-medlegal"
HF_REPO     = "Cukinator/qwen25-14b-medlegal"
HF_TOKEN    = UserSecretsClient().get_secret("HF_TOKEN")

gc.collect()
torch.cuda.empty_cache()
print(f"🧹 VRAM libre antes del merge: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

print("📥 Cargando modelo + adapters con Unsloth...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=ADAPTER_DIR,
    max_seq_length=512,
    dtype=torch.float16,
    load_in_4bit=True,
)

print(f"🚀 Mergeando y subiendo a {HF_REPO}...")
model.push_to_hub_merged(
    HF_REPO,
    tokenizer,
    save_method="merged_16bit",
    token=HF_TOKEN,
)
print(f"✅ Completado → https://huggingface.co/{HF_REPO}")

🧹 VRAM libre antes del merge: 2.5 GB
📥 Cargando modelo + adapters con Unsloth...
==((====))==  Unsloth 2026.3.4: Fast Qwen2 patching. Transformers: 5.2.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

unsloth/qwen2.5-14b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.
🚀 Mergeando y subiendo a Cukinator/qwen25-14b-medlegal...
Unsloth: Saving to Cukinator/qwen25-14b-medlegal will fail, but using a temp folder works! Switching to a temp folder then uploading!


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

...


Unsloth: Merging weights into 16bit:  83%|████████▎ | 5/6 [08:36<01:44, 104.01s/it]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


Unsloth: Merging weights into 16bit: 100%|██████████| 6/6 [10:08<00:00, 101.39s/it]


Unsloth: Merge process complete. Saved to `/tmp/tmppqlpc_0z`
✅ Completado → https://huggingface.co/Cukinator/qwen25-14b-medlegal


In [22]:
# Correr en celda separada para fixear el repo
from huggingface_hub import HfApi
import json, tempfile, os

api = HfApi()

# Descargar config del base
from huggingface_hub import hf_hub_download
base_config_path = hf_hub_download(BASE_MODEL, "config.json")
with open(base_config_path) as f:
    base_config = json.load(f)

# Subir al repo
with tempfile.NamedTemporaryFile("w", suffix=".json", delete=False) as tmp:
    json.dump(base_config, tmp, indent=2)
    tmp_path = tmp.name

api.upload_file(
    path_or_fileobj=tmp_path,
    path_in_repo="config.json",
    repo_id=REPO_ID,
)
os.unlink(tmp_path)
print("✅ config.json actualizado en el repo")

✅ config.json actualizado en el repo


In [6]:
from huggingface_hub import login, HfApi
from kaggle_secrets import UserSecretsClient
import os, json

# Re-login forzado
login(token=UserSecretsClient().get_secret("HF_TOKEN"), add_to_git_credential=False)

# Verificar identidad
api = HfApi()
print("✅ Logueado como:", api.whoami()["name"])

# Subir
os.makedirs("/tmp/gen_config", exist_ok=True)
with open("/tmp/gen_config/generation_config.json", "w") as f:
    json.dump({"max_new_tokens": 512, "temperature": 0.7, "do_sample": True}, f, indent=2)

api.upload_file(
    path_or_fileobj="/tmp/gen_config/generation_config.json",
    path_in_repo="generation_config.json",
    repo_id="Cukinator/qwen25-14b-medlegal",
    commit_message="Fix generation_config: set max_new_tokens=512",
)
print("✅ Subido a HF")

✅ Logueado como: Cukinator
✅ Subido a HF


# Now testing time (reset enviroment and reinstall dependencies)

In [7]:
# ─────────────────────────────────────────────
# CELDA  — Cargar modelo
# ─────────────────────────────────────────────
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login, snapshot_download
from kaggle_secrets import UserSecretsClient

login(token=UserSecretsClient().get_secret("HF_TOKEN"))

BASE_MODEL = "Qwen/Qwen2.5-14B-Instruct"
REPO_ID    = "Cukinator/qwen25-14b-medlegal"

print("📥 Cargando tokenizer...")
tokenizer_eval = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)

print("📥 Descargando modelo (excluyendo adapter y checkpoints)...")
model_path = snapshot_download(
    REPO_ID,
    ignore_patterns=[
        "adapter_config.json",
        "adapter_model.safetensors",
        "checkpoint-*/",
    ],
)
print(f"✅ Descargado en: {model_path}")

print("📥 Cargando modelo (~29GB, puede tardar varios minutos)...")
model_eval = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model_eval.eval()
print("✅ Modelo cargado correctamente")
print(f"   VRAM usada: {torch.cuda.memory_allocated()/1e9:.1f} GB")

📥 Cargando tokenizer...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


✅ Descargado en: /root/.cache/huggingface/hub/models--Cukinator--qwen25-14b-medlegal/snapshots/e3ad88ab12dc4ecb7da44284b86068d1a75f5424
📥 Cargando modelo (~29GB, puede tardar varios minutos)...


Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

[accelerate.big_modeling|WARNING]Some parameters are on the meta device because they were offloaded to the cpu.


✅ Modelo cargado correctamente
   VRAM usada: 13.7 GB


In [20]:
import torch

model_eval.eval()

SYSTEM_MEDICO = (
    "Sos un médico clínico experto con más de 20 años de experiencia. "
    "SIEMPRE respondés en español, nunca en inglés. "
    "Respondés preguntas médicas de forma detallada, con al menos 3-4 párrafos. "
    "Explicás causas posibles, síntomas asociados, qué hacer y qué evitar. "
    "Siempre recomendás consultar a un profesional de la salud para diagnósticos definitivos."
)
SYSTEM_LEGAL = (
    "Sos un abogado experto en derecho con más de 20 años de experiencia. "
    "SIEMPRE respondés en español, nunca en inglés. "
    "Respondés consultas legales de forma clara y precisa, con al menos 3-4 párrafos. "
    "Siempre aclarás que tu respuesta es orientativa y recomendás consultar a un abogado matriculado."
)

test_cases = [
    (SYSTEM_MEDICO, "Tengo fiebre de 38.5°C hace 3 días con dolor de garganta, ¿qué puede ser?"),
    (SYSTEM_MEDICO, "¿Cuál es la diferencia entre diabetes tipo 1 y tipo 2?"),
    (SYSTEM_LEGAL,  "Mi empleador no me pagó el sueldo de este mes, ¿qué puedo hacer?"),
    (SYSTEM_LEGAL,  "¿Qué diferencia hay entre un contrato laboral y uno de locación de servicios?"),
]

eos_id = tokenizer_eval.convert_tokens_to_ids("<|im_end|>")
print(f"eos_id: {eos_id}\n")

print("🗣️ Respuestas del modelo médico-legal:\n")
for system, prompt in test_cases:
    role = "🏥 MÉDICO" if "médico" in system else "⚖️ LEGAL"

    messages = [
        {"role": "system", "content": system},
        {"role": "user",   "content": prompt},
    ]
    text = tokenizer_eval.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = tokenizer_eval(text, return_tensors="pt").to(model_eval.device)

    with torch.no_grad():
        output_ids = model_eval.generate(
            **inputs,
            max_new_tokens=1024,
            do_sample=True,
            temperature=0.7,
            repetition_penalty=1.1,
            eos_token_id=eos_id,
            pad_token_id=tokenizer_eval.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    print(f"🔢 Tokens generados: {len(new_tokens)}")
    print(f"🔢 Último token id: {new_tokens[-1].item()} | eos_id esperado: {eos_id}")

    response_raw = tokenizer_eval.decode(new_tokens, skip_special_tokens=False)
    response_clean = tokenizer_eval.decode(new_tokens, skip_special_tokens=True)

    print(f"{role} ❓ {prompt}")
    print(f"💬 Raw:   {response_raw}")
    print(f"💬 Clean: {response_clean}")
    print("-" * 60)

eos_id: 151645

🗣️ Respuestas del modelo médico-legal:

🔢 Tokens generados: 21
🔢 Último token id: 151645 | eos_id esperado: 151645
🏥 MÉDICO ❓ Tengo fiebre de 38.5°C hace 3 días con dolor de garganta, ¿qué puede ser?
💬 Raw:   Puede ser una infección viral o bacteriana. Si persiste consulta a tu médico.<|im_end|>
💬 Clean: Puede ser una infección viral o bacteriana. Si persiste consulta a tu médico.
------------------------------------------------------------
🔢 Tokens generados: 27
🔢 Último token id: 151645 | eos_id esperado: 151645
🏥 MÉDICO ❓ ¿Cuál es la diferencia entre diabetes tipo 1 y tipo 2?
💬 Raw:   Tipo 1 es autoinmune; tipo 2 está relacionado con factores de riesgo como obesidad.<|im_end|>
💬 Clean: Tipo 1 es autoinmune; tipo 2 está relacionado con factores de riesgo como obesidad.
------------------------------------------------------------
🔢 Tokens generados: 28
🔢 Último token id: 151645 | eos_id esperado: 151645
⚖️ LEGAL ❓ Mi empleador no me pagó el sueldo de este mes, ¿qué pue